<a id="unit-u.af358f5a6eef43fe8292980b5082426e"></a>

<a id="analyze-traces-with-sql-and-litefuse"></a>

# Трассаларды SQL және Litefuse арқылы талдау

**7-модуль: Трассаларды талдау және Litefuse** · [Сабақ](course.md) · [Тест](quiz07_analysis_and_litefuse.ipynb)

Бұл жұмыста трассаларды екі жолмен талдайсыз. Алдымен демо сервистері `otel_traces` ішіне жазатын span-дарда SQL-мен төлем инцидентін өлшейсіз: әр операцияның сәтсіздік үлесі мен кідірісі, сәтсіз тапсырыстардың басқаларынан айырмасы, әр ақау басталатын сервис және сәтсіз үлестің уақыт бойынша өзгерісі. Кейін үлкен тілдік модель (LLM) негізіндегі шағын қолданбаны дерегін Doris-те сақтайтын LLM бақылау платформасы Litefuse арқылы трассалайсыз. Трассаларды Langfuse Python SDK-мен жіберіп, Doris-тен SQL-мен тауып, сол трассаларды Litefuse веб-интерфейсінде ашасыз.

Қолданба ноутбуктегі имитациялық модельді шақырады: API кілті не тіркелгі қажет емес, ештеңе компьютеріңізден шықпайды.

Шамамен 50 минут: 1–2-бөлімдерге 20, 3–9-бөлімдерге 30 минут жоспарлаңыз. Төлем ақауы қосылған 1-модульдің DOG стегі мен демосы қажет (1-зертханалық жұмыстың 5-бөлімі). 4-бөлім Litefuse іске қосады; алғашында шамамен 1.2 GB бейне жүктеліп, тағы бірнеше минут алады.

<a id="unit-u.c8581b16304b49ee8b0b4e1c6907c628"></a>

<a id="initialize-the-lab"></a>

### Зертханалық жұмысты инициализациялау

`lab.shell(...)` не `lab.sql(...)` қолданбас бұрын осы ұяшықты бір рет орындаңыз.


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT)


<a id="unit-u.893be45c0c5849539da6191dd7c1340c"></a>

<a id="1-check-the-spans-in-doris"></a>

## 1. Doris-тегі span-дарды тексеру

Төмендегі сұраулар сервер span-дарын реттеп қорытындылайды. Сервер span-ы бір кіріс сұрауға сервистің істеген жұмысын қамтиды; оның `span_kind` мәні — `SPAN_KIND_SERVER`. Сервис бойынша санаңыз:

In [ ]:
server_spans = lab.wait_for_rows(
    """
    SELECT service_name, COUNT(*) AS server_spans, MAX(timestamp) AS latest_span
    FROM otel_traces
    WHERE span_kind = 'SPAN_KIND_SERVER'
    GROUP BY service_name
    ORDER BY server_spans DESC
    """,
    title="Server spans by service in otel_traces",
    hint="otel_traces has no server spans. Start the demo with lab.start_demo() in Module 1, "
         "wait a minute, then rerun this cell.",
)

<a id="unit-u.07078eed6b9b4bb7a92e6ed12b38ec54"></a>

**Күтілетін нәтиже**

Сұрау өңдейтін әр сервиске бір жол, демо бірнеше минут істегенде шамамен 14. `frontend-proxy` мен `frontend` сервер span-дары ең көп; `checkout`, `payment`, `email` аз: жүктеме генераторы тапсырыс бергеннен әлдеқайда жиі дүкенді қарайды. 2-модульде жіберген span жоқ: ол сервер емес, ішкі span.

<a id="unit-u.43be7881141b405bb92d16e62f321f16"></a>

<a id="2-measure-the-incident-with-sql"></a>

## 2. Инцидентті SQL-мен өлшеу

3-модульде инцидентті бір-бір сұраудан қадағаладыңыз. `otel_traces` бойынша SQL бәрін бірден өлшейді. Бұл бөлім сұраулары алғашқы сәтсіз тапсырыстан қазіргі сәтке дейінгі инцидент терезесін қарайды. Бастауын, кейін сол терезеде сәтсіз сұраулары бар сервер операцияларын табыңыз: әрқайсысы қанша сұрау өңдеді, қаншасы және қандай үлесі сәтсіз, қанша уақыт алды.

In [ ]:
incident = lab.wait_for_rows(
    """
    SELECT MIN(timestamp) AS first_failed_order, COUNT(*) AS failed_orders
    FROM otel_traces
    WHERE service_name = 'checkout'
      AND span_name = 'oteldemo.CheckoutService/PlaceOrder'
      AND status_code = 'STATUS_CODE_ERROR'
    HAVING COUNT(*) > 0
    """,
    title="The failed orders in otel_traces",
    hint="No order has failed. Check that the demo runs (lab.start_demo() in Module 1) and that paymentFailure "
         "is 50% (section 5 of Lab 1), wait a minute or two, then rerun this cell.",
    timeout_seconds=180,
)
INCIDENT_START = str(incident["first_failed_order"][0])

lab.sql(f"""
SELECT service_name, span_name,
       COUNT(*) AS requests,
       SUM(IF(status_code = 'STATUS_CODE_ERROR', 1, 0)) AS failed,
       ROUND(100 * SUM(IF(status_code = 'STATUS_CODE_ERROR', 1, 0)) / COUNT(*), 1) AS failed_percent,
       ROUND(PERCENTILE_APPROX(duration, 0.5) / 1000.0, 2) AS p50_ms,
       ROUND(PERCENTILE_APPROX(duration, 0.99) / 1000.0, 2) AS p99_ms
FROM otel_traces
WHERE span_kind = 'SPAN_KIND_SERVER'
  AND timestamp >= '{INCIDENT_START}'
GROUP BY service_name, span_name
HAVING SUM(IF(status_code = 'STATUS_CODE_ERROR', 1, 0)) > 0
ORDER BY failed_percent DESC
""", title="Server operations with failed requests since the first failed order");

<a id="unit-u.6be9f4ff94a84d45b74e146ed00277dd"></a>

**Күтілетін нәтиже**

Алдымен инцидент терезесінің басы мен сәтсіз тапсырыс саны. Кейін әдетте төрт жол:

- `checkout` `oteldemo.CheckoutService/PlaceOrder` және `payment` `grpc.oteldemo.PaymentService/Charge`: сұрау және ақау саны бірдей, шамамен жартысы сәтсіз. Аз тапсырыста кездейсоқтық үлесті жартыдан алыс етуі мүмкін. Әр тапсырыс бір төлем жасайды, сәтсіз төлем тапсырысын сәтсіз етеді.
- `frontend` `POST`: көбірек сұраудың ішіндегі сол ақаулар; себетке тауар қосу сияқты басқа POST сұрауларында сол span атауы бар.
- `frontend-proxy` `ingress`: ондаған мың сұраудың ішіндегі сол ақаулар, сәтсіздік үлесі 0-ге жуық. Шлюз үлесі бір операция ақауын басқалардың арасында жасырады; үлесті операциямен бөліңіз.

`p50_ms` — медиана, `p99_ms` — 99-процентиль: сұраулардың 99%-ы ең көбі сонша уақыт алды. `PERCENTILE_APPROX` бәрін сұрыптамай, ұзақтықтардың ықшам қорытындысынан есептейді, кідіріске жеткілікті дәл. `duration` микросекунд сақтайды, сондықтан сұрау 1,000-ға бөледі. Бірнеше ондаған тапсырыста жалғыз баяу тапсырыс `PlaceOrder` P99-ын анықтайды; сұрау көбейгенде процентиль мағыналырақ.

<a id="unit-u.71b6e59c238e42c28f7857c3a912418d"></a>

<a id="failed-orders-and-successful-orders"></a>

### Сәтсіз және сәтті тапсырыстар

Сәтсіз тапсырыстар ұзақтығын сәттілерімен салыстырыңыз:

In [ ]:
lab.sql(f"""
SELECT IF(status_code = 'STATUS_CODE_ERROR', 'failed', 'succeeded') AS outcome,
       COUNT(*) AS orders,
       ROUND(PERCENTILE_APPROX(duration, 0.5) / 1000.0, 1) AS p50_ms,
       ROUND(PERCENTILE_APPROX(duration, 0.99) / 1000.0, 1) AS p99_ms,
       ROUND(MAX(duration) / 1000.0, 1) AS max_ms
FROM otel_traces
WHERE service_name = 'checkout'
  AND span_name = 'oteldemo.CheckoutService/PlaceOrder'
  AND timestamp >= '{INCIDENT_START}'
GROUP BY outcome
""", title="The duration of the orders since the first failed order, by outcome");

<a id="unit-u.597a3cb2bd2d4b24b938dc13ab5fa24a"></a>

**Күтілетін нәтиже**

Екі жол. Сәтсіз тапсырыстар жылдамырақ: медианасы сәттілерден әлдеқайда төмен, мысалы 11 ms орнына 6 ms. Тапсырыс төлемде сәтсіз аяқталады; сәттісін checkout жалғастырып жөнелтеді, себетті босатады, растау хатын жібереді, Kafka-ға жариялайды. Ақау әрдайым сұрауды баяулатпайды; тек кідіріс ескертуі бұл инцидентті көрмес еді.

<a id="unit-u.11000248a8e84aa88cc19f28679ec693"></a>

<a id="where-each-failure-starts"></a>

### Әр ақау қайдан басталады

3-модульде бір ақау бастауын трассадан, ең терең қате span-ынан таптыңыз. Әр сәтсіз тапсырыс үшін SQL-мен табу үшін трасса сервистерін салыстырыңыз. Сервер span-ы сәтсіз, ал басқа сервистерге өз шақыруларының ешқайсысы сәтсіз емес сервис — бастау: ол басқаның кесірінен істен шықпады. Келесі сұрау сәтсіз тапсырыс трассаларынан сондай сервер span-дарын `NOT EXISTS` арқылы табады; ол ішкі сұрау сәйкес жол таппаса ғана қалдырады. Барлық span-ы Doris-ке әлі жетпеуі мүмкін соңғы минут тапсырыстарын қоспайды.

In [ ]:
lab.sql(f"""
SELECT s.service_name, s.span_name, COUNT(DISTINCT s.trace_id) AS failed_orders
FROM otel_traces AS s
WHERE s.span_kind = 'SPAN_KIND_SERVER'
  AND s.status_code = 'STATUS_CODE_ERROR'
  AND s.timestamp >= '{INCIDENT_START}'
  AND s.trace_id IN (
      SELECT trace_id
      FROM otel_traces
      WHERE service_name = 'checkout'
        AND span_name = 'oteldemo.CheckoutService/PlaceOrder'
        AND status_code = 'STATUS_CODE_ERROR'
        AND timestamp >= '{INCIDENT_START}'
        AND end_time < UTC_TIMESTAMP() - INTERVAL 1 MINUTE
  )
  AND NOT EXISTS (
      SELECT 1
      FROM otel_traces AS c
      WHERE c.trace_id = s.trace_id
        AND c.service_name = s.service_name
        AND c.span_kind = 'SPAN_KIND_CLIENT'
        AND c.status_code = 'STATUS_CODE_ERROR'
  )
GROUP BY s.service_name, s.span_name
""", title="Server spans that failed without a failed call, in the traces of the failed orders");

<a id="unit-u.1d043f1b144b4d57a335bd18ac797f38"></a>

**Күтілетін нәтиже**

Бір жол: `payment`, `grpc.oteldemo.PaymentService/Charge`; саны бір минуттан ескі барлық сәтсіз тапсырысқа тең. Жолдағы `checkout`, `frontend`, `frontend-proxy` сервистерінің әрқайсысында сол трассада өз сәтсіз шақыруы бар. Сондықтан ішкі сұрау жол тауып, `NOT EXISTS` алып тастайды: олар payment кесірінен сәтсіз болған.

Сұрау әдейі жеке span-дарды емес, сервистерді салыстырады. Frontend ішінде `POST /api/checkout` екі қате span-ының арасында өзі қате тіркемейді; балалары қатесіз қате span-ын іздейтін ереже frontend `POST` span-ында да тоқтайды. Трасса span-дары сағаттары мен SDK-лары миллисекунд бөліктеріне келіспейтін әртүрлі сервистен келеді, сондықтан ерте аяқталған span әрдайым бірінші сәтсіз болғаны емес.

<a id="unit-u.9733af5465bb4d759a9af02278604462"></a>

<a id="the-share-of-failed-orders-over-time"></a>

### Сәтсіз тапсырыстар үлесінің уақыт бойынша өзгерісі

1-зертханалық жұмыс тапсырыстарды минутпен санап, минутына бір-үш тапсырыста бір минут аз мәлімет беретінін байқады. Терезелік функция әр тапсырысқа соңғы он тапсырыстың сәтсіздік үлесін есептей алады. `GROUP BY` орнына ол әр жолды сақтап, өзімен байланысты терезеден мән есептейді: мұнда уақыт ретімен тапсырыстар, `AVG` үшін ағымдағы және алдыңғы тоғыздан тұратын жақтау.

In [ ]:
lab.sql("""
SELECT order_number, ordered_at, outcome, failed_share_of_last_10
FROM (
    SELECT ROW_NUMBER() OVER (ORDER BY timestamp) AS order_number,
           timestamp AS ordered_at,
           IF(status_code = 'STATUS_CODE_ERROR', 'failed', 'succeeded') AS outcome,
           ROUND(AVG(IF(status_code = 'STATUS_CODE_ERROR', 1, 0))
                 OVER (ORDER BY timestamp ROWS BETWEEN 9 PRECEDING AND CURRENT ROW), 2) AS failed_share_of_last_10
    FROM otel_traces
    WHERE service_name = 'checkout'
      AND span_name = 'oteldemo.CheckoutService/PlaceOrder'
) AS orders
ORDER BY order_number DESC
LIMIT 15
""", title="The latest orders, with the share of failures among the last ten");

<a id="unit-u.b16d41d9d996499a9c1f4376d19ac608"></a>

**Күтілетін нәтиже**

15 жолға дейін, алдымен соңғы тапсырыс. `ROW_NUMBER()` демо басталғалы барлық тапсырысты уақыт ретімен нөмірлейді, сондықтан соңғысының нөмірі ең үлкен. `failed_share_of_last_10` шамамен 0.5, 0.3–0.7 арасында болып тұрады, жеке тапсырыстар кездейсоқ `failed` не `succeeded` болады. Жалаушаны қосқаннан кейінгі алғашқы жақтауларда бұрынғы тапсырыстар да бар, үлес төменірек; демо аз тапсырыс берсе, жақтау тек соларды қамтып, кездейсоқтық үлесті 0.5-тен алыс етуі мүмкін. `PARTITION BY` нөмірлеу мен жақтауларды, мысалы әр сервиске, қайта бастар еді; 8-модуль осындай үлеске ескертеді.

<a id="unit-u.d93aa3b23a414ffab9fdb2a6439a9a5c"></a>

<a id="3-litefuse-traces-of-an-llm-application"></a>

## 3. Litefuse: LLM қолданбасының трассалары

`otel_traces` span-дары сервистер арасындағы сұрауларды сипаттайды: қайсысы қайсысын шақырды, әр шақыру қанша уақыт алды. LLM қолданбасына көбірек мәлімет керек: модельге жіберген промпт, жауабы, модель санаған токендер мен құны, промптқа құжат іздеу сияқты айналасындағы қадамдар. LLM бақылау платформалары да әр сұрауға трасса, әр қадамға бақылау жазбасын тіркейді.

Litefuse — осындай платформа, Langfuse-тің ашық форкі ([github.com/litefuse/litefuse](https://github.com/litefuse/litefuse), MIT лицензиясы): Langfuse трассаларын ClickHouse-та, Litefuse Apache Doris-те сақтайды. API-ы Langfuse-пен үйлесімді, сондықтан қолданбалар Litefuse адресіне бағытталған Langfuse SDK-ларымен трасса жібереді. Бұл курста Litefuse кестелері DOG стегінің Doris-інде, `otel` жанындағы `litefuse` дерекқорында.

Litefuse 26.3.0 бес контейнермен жұмыс істейді:

| Контейнер | Міндеті |
| --- | --- |
| `doris-obs-litefuse-web` | Веб-интерфейс және трасса қабылдайтын API |
| `doris-obs-litefuse-worker` | Трассаларды фонда Doris-ке жүктейді |
| `doris-obs-litefuse-postgres` | PostgreSQL: пайдаланушылар, ұйымдар, жобалар, API кілттері, промпттар |
| `doris-obs-litefuse-redis` | Redis: web контейнері мен worker арасындағы кезектер |
| `doris-obs-litefuse-silo` | MinIO-дан тармақталған S3-үйлесімді объектілік сақтау сервері Silo: API алған әр трасса пакеті |

API әр пакетті объектілік сақтауға сақтап, кезекке қояды. Worker кезектегілерді Stream Load арқылы Doris-ке жүктейді, интерфейс пен API SQL-мен қайта оқиды.

<a id="unit-u.583124c1b65e4bf9aa0f6bed15b63258"></a>

<a id="4-start-litefuse"></a>

## 4. Litefuse іске қосу

`lab.start_litefuse()` Litefuse-ті әр бейне нұсқасы бекітілген `environments/litefuse/compose.litefuse.yml` файлымен іске қосады. Ол пайдаланушымен, `Doris course` ұйымымен, API кілт жұбы бар `LLM app` жобасымен басталады, тіркелу қажет емес.

Контейнер бастамас бұрын, Litefuse жүктей алуы үшін ұяшық Doris-те бір команда орындайды:

- Litefuse Stream Load арқылы FE `doris-obs:8030` адресімен жүктейді. FE деректі өзі қабылдамайды: BE тіркелген адреске бағыттау жауабын береді.
- Бір контейнерлі Doris BE-сі `127.0.0.1` деп тіркелген. Басқа контейнер үшін бұл — өзінің адресі, жүктеу «connection refused» болып сәтсіз болады.
- `ALTER SYSTEM MODIFY BACKEND` BE-ге `private_endpoint` тегімен жеке соңғы нүкте береді. Ол бар болса, FE Stream Load-ты клиент FE-ге жетуге қолданған хостқа, мұнда `doris-obs`, сол нүктенің 8040 портымен бағыттайды. `doris-obs` желісінің контейнерлері BE-ге сол жерден жетеді. [Күрделі желілік ортадағы Stream Load тәжірибелері](https://doris.apache.org/docs/4.x/data-operate/import/load-best-practices/stream-load-in-complex-network/) соңғы нүктелерді сипаттайды.

In [ ]:
lab.start_litefuse()

<a id="unit-u.7e7a19ecb60d45c0bc402741dfc0e7c6"></a>

**Күтілетін нәтиже**

Ретімен:

- Порт тексеруі: 23100 (`DORIS_OBS_LITEFUSE_PORT`) — `available`, Litefuse қосулы болса `used by this course`.
- `location` және `private_endpoint` тегтері бар `ALTER SYSTEM MODIFY BACKEND "127.0.0.1:9050"` командасы, «Statement succeeded». `SHOW BACKENDS` енді `Tag` бағанында екеуін көрсетеді.
- `docker compose` журналы: алғашында жүктеулер, кейін `ps` жұмыс істейді деп тізетін бес `doris-obs-litefuse-*` контейнері; Postgres, Redis, Silo healthy деп те белгіленген.
- API кілттері тиесілі жоба: `Doris course` ұйымы, `LLM app` жобасы және `llm_app` ID-і.
- Контейнерлерден шамамен 10 секунд кейін «Litefuse is running» карточкасы: алғашында Litefuse жауап бермес бұрын Postgres дерекқорын және Doris кестелерін құрады.

Қайталау жұмыс істейтін контейнерлер мен деректерін сақтайды.

<a id="unit-u.629c2edd67484bc4ad73997e8d2f8cb2"></a>

<a id="5-trace-a-small-llm-application"></a>

## 5. Шағын LLM қолданбасын трассалау

Келесі ұяшық курс туралы сұрақтарға жауап беретін шағын көмекші құрып, Langfuse Python SDK-мен (`langfuse`, курс талаптарымен орнатылады) трассалайды. Әр қадам — әр шақыруды бақылау жазбасы ретінде, функция аргументтерін кіріс, қайтарған мәнін шығыс етіп тіркейтін `@observe` декораторы бар функция:

- `answer_question` бір сұрауды өңдейді, трасса түбірі болады. `propagate_attributes` трассаны `course-assistant` деп атап, сұраған пайдаланушы мен сеансты тіркейді; төмендегі жазбалар соларды мұралайды.
- `search_docs` сұрақ тақырыбының құжаттарын іздейді: түрі `retriever` бақылау жазбасы.
- `call_model` модель шақыруын білдіреді: түрі `generation`. Имитациялық модель дайын жауап қайтарады, сөзге бір токен санап, секундына 100 токен жазады. `update_current_generation` промптты (чат хабарларын), модельді, токен санын және мысалдағы 1,000 кіріс токенге 0.01 USD, 1,000 шығыс токенге 0.03 USD бағасынан шақыру құнын тіркейді. Нақты модельде OpenAI SDK интеграциясы сияқты Langfuse интеграциялары бұларды өздері тіркейді: [Қосымша 7A](optional07_trace_a_real_model.ipynb) көмекшіге компьютеріңіздегі не OpenAI-дағы нақты модель береді.

SDK API кілттері мен Litefuse адресін `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY`, `LANGFUSE_BASE_URL` айнымалыларынан оқиды. Бақылау жазбаларын OpenTelemetry span-дары ретінде пакетпен Litefuse-тің `/api/public/otel/v1/traces` соңғы нүктесіне жібереді; `flush()` ұяшық бітпес бұрын қалғанын жібереді. `OTEL_SERVICE_NAME` демо сервистері `otel_traces` ішінде аталғандай, span ресурстарында қолданбаны атайды.

In [ ]:
import os
import time

os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-doris-course"
os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-doris-course"
os.environ["LANGFUSE_BASE_URL"] = lab.litefuse_url
os.environ["OTEL_SERVICE_NAME"] = "course-assistant"

from langfuse import get_client, observe, propagate_attributes

langfuse = get_client()

# The documents that the assistant searches, by topic, and the answers of the simulated model.
DOCUMENTS = {
    "doris": "Apache Doris is a real-time analytical database. It answers SQL queries over large tables in seconds.",
    "litefuse": "Litefuse is an LLM engineering platform. It keeps its traces and observations in Apache Doris tables.",
    "collector": "The OpenTelemetry Collector receives logs, metrics, and traces and exports them to Doris.",
}
ANSWERS = {
    "What is Apache Doris?":
        "Apache Doris is a real-time analytical database that answers SQL queries over large tables in seconds.",
    "Where does Litefuse keep its traces?":
        "Litefuse keeps its traces and observations in Apache Doris tables, so you can query them with SQL.",
}
PRICE_PER_1K_TOKENS = {"input": 0.01, "output": 0.03}  # an example price list, in USD


def count_tokens(text):
    return len(text.split())  # the simulated model counts one token per word


@observe(as_type="retriever", name="search-docs")
def search_docs(question):
    time.sleep(0.05)  # a lookup in a document index takes a moment
    return [text for topic, text in DOCUMENTS.items() if topic in question.lower()]


@observe(as_type="generation", name="llm-answer")
def call_model(question, documents):
    messages = [
        {"role": "system", "content": "Answer the question with these documents: " + " ".join(documents)},
        {"role": "user", "content": question},
    ]
    answer = ANSWERS[question]
    usage = {"input": sum(count_tokens(message["content"]) for message in messages),
             "output": count_tokens(answer)}
    time.sleep(usage["output"] / 100)  # the simulated model writes 100 tokens per second
    langfuse.update_current_generation(
        input=messages,
        model="simulated-llm",
        model_parameters={"temperature": 0},
        usage_details=usage,
        cost_details={kind: round(tokens / 1000 * PRICE_PER_1K_TOKENS[kind], 6) for kind, tokens in usage.items()},
    )
    return answer


@observe(name="answer-question")
def answer_question(question, *, user_id, session_id):
    with propagate_attributes(trace_name="course-assistant", user_id=user_id, session_id=session_id):
        trace_ids.append(langfuse.get_current_trace_id())
        return call_model(question, search_docs(question))


trace_ids = []
for user_id, session_id, question in [
    ("user-1", "session-1", "What is Apache Doris?"),
    ("user-1", "session-1", "Where does Litefuse keep its traces?"),
    ("user-2", "session-2", "What is Apache Doris?"),
]:
    print(f"{user_id}: {question}\n  -> {answer_question(question, user_id=user_id, session_id=session_id)}")
langfuse.flush()
print("Trace IDs:", trace_ids)

<a id="unit-u.4312e012ca0844e89b616431c3ea1929"></a>

**Күтілетін нәтиже**

Жауаптарымен үш сұрақ: екеуі `user-1` / `session-1`, біреуі `user-2` / `session-2`; кейін `Trace IDs:` және 32 он алтылық таңбалы үш ID — `otel_traces` ішіндегідей OpenTelemetry трасса ID-лері. Келесі бөлімдер осы үшеуін іздейді. Ұяшықты қайталау үш жаңа трасса жібереді, келесі бөлімдер енді жаңаларын көрсетеді.

<a id="unit-u.1f223c6169734a368a8e5ab323f36406"></a>

<a id="6-find-the-traces-in-doris"></a>

## 6. Трассаларды Doris-тен табу

Жобаның алғашқы трассасы келгенде Litefuse `llm_app` жоба ID-імен аталған екі кесте құрады: `spans_llm_app` — бақылау жазбасына бір жол, `traces_scalar_llm_app` — трассаға бір жол. Келесі ұяшық оларды күтеді.

In [ ]:
lab.wait_for_rows(
    """
    SELECT TABLE_NAME AS table_name, TABLE_TYPE AS table_type
    FROM information_schema.tables
    WHERE TABLE_SCHEMA = 'litefuse' AND TABLE_NAME LIKE '%llm_app'
    ORDER BY TABLE_NAME
    """,
    title="The tables of the project llm_app",
    hint="Litefuse creates them when the project's first trace arrives. Check that the previous cell printed "
         "three trace IDs and that Litefuse runs (section 4). The worker's log, docker logs "
         "doris-obs-litefuse-worker, shows the creation of the tables. Then rerun this cell.",
    min_rows=2,
);

<a id="unit-u.c55a171eed004b40b38a3d8b7f983c90"></a>

**Күтілетін нәтиже**

Алғашқы трассалардан бірнеше секунд кейін `spans_llm_app` және `traces_scalar_llm_app` деген екі жол, екеуі де `BASE TABLE`.

In [ ]:
lab.sql("SHOW TABLES FROM litefuse", title="Litefuse's tables in Doris");

<a id="unit-u.95b044343b2d4b9bb46398f1e866f824"></a>

**Күтілетін нәтиже**

Сегіз кесте. Жобаның екеуінен бөлек Litefuse алғашында алтауын құрған: `scores` — пайдаланушы пікірі не бағалау нәтижесі сияқты трасса бағалары; `dataset_run_items_rmt` — деректер жиынындағы тәжірибе нәтижелері; `event_log` пен `blob_storage_file_log` — API алған оқиғалардың объектілік сақтаудағы орны; `project_environments` — әр жоба орталары; `schema_migrations` — Litefuse дерекқорға қолданған миграциялар. Әр жоба span-дары мен трассаларына жеке жұп кесте алады.

<a id="unit-u.832f7ce19ae9467da6455f60f6aa46d8"></a>

Келесі ұяшық үш трассаның бақылау жазбаларын күтеді. `usage_details` — түрі бойынша токен санын сақтайтын `MAP`; `TIMESTAMPDIFF` басталу/аяқталу уақытынан әр жазба ұзақтығын береді.

In [ ]:
TRACE_ID_LIST = ", ".join(f"'{trace_id}'" for trace_id in trace_ids)
llm_spans = lab.wait_for_rows(
    f"""
    SELECT trace_id, user_id, name, type, provided_model_name AS model, usage_details, total_cost,
           TIMESTAMPDIFF(MICROSECOND, start_time, end_time) / 1000 AS duration_ms
    FROM litefuse.spans_llm_app
    WHERE trace_id IN ({TRACE_ID_LIST})
    ORDER BY start_time, is_root DESC
    """,
    title="The observations of the three traces in spans_llm_app",
    hint="Litefuse loads a batch of traces into Doris a few seconds after it receives it. Check the worker's "
         "log with docker logs doris-obs-litefuse-worker, then rerun this cell.",
    min_rows=9,
    timeout_seconds=90,
)

<a id="unit-u.866aff099d404b59817d8ad6158f1f49"></a>

**Күтілетін нәтиже**

Сұрау ретімен тоғыз жол, трассаға үшеуден. Алғашқыда жібергеннен шамамен 10 секунд кейін келеді: Litefuse worker-і жаңа жоба кестелеріне жүктемес бұрын сонша күтеді. Кейін бірнеше секунд алады. Doris туралы сұрақ үшін:

| `name` | `type` | `model` | `usage_details` | `total_cost` | `duration_ms` |
| --- | --- | --- | --- | --- | --- |
| `answer-question` | `SPAN` | NULL | `{}` | NULL | шамамен 220 |
| `search-docs` | `RETRIEVER` | NULL | `{}` | NULL | шамамен 55 |
| `llm-answer` | `GENERATION` | `simulated-llm` | `{"input":26, "output":16, "total":42}` | `0.000740000000` | шамамен 160 |

Litefuse туралы сұрақта `{"input":28, "output":17, "total":45}` және `0.000790000000` бар. Litefuse жиынтықтарды: `usage_details` ішіндегі `total` және `total_cost` мәндерін қосты. Әр трассаның түбір span-ы шамамен екі қадамының қосындысынша уақыт алады; әр жолда `propagate_attributes` орнатқан `user_id` бар.

In [ ]:
lab.sql(f"""
SELECT input, output
FROM litefuse.spans_llm_app
WHERE trace_id = '{trace_ids[1]}' AND type = 'GENERATION'
""", title="The prompt and the answer of the second request");

<a id="unit-u.6caafff486a644a2a677f55ef94ae20f"></a>

**Күтілетін нәтиже**

Бір жол. `input` — промпт: JSON түріндегі екі чат хабары, `search_docs` тапқан Litefuse құжаты бар жүйелік хабар және пайдаланушы сұрағы. `output` — жауап. Қалған бақылау жазбалары сол бағандарда функция аргументтері мен қайтарған мәнін сақтайды.

<a id="unit-u.468fd46bbb28408a8254fa7ec9db1cca"></a>

<a id="7-the-same-traces-in-litefuse"></a>

## 7. Сол трассалар Litefuse ішінде

Litefuse веб-интерфейсін ашыңыз:

1. [http://127.0.0.1:23100](http://127.0.0.1:23100) адресіне (не `DORIS_OBS_LITEFUSE_PORT` портына) өтіңіз. Төмендегі мәзір атауларына сәйкес болу үшін кіру бетінің тіл тізімінен **English** таңдап, `learner@example.com` поштасымен, `doris-course-1` құпиясөзімен кіріңіз.
2. Бет **LLM app** жобасы бар **Doris course** ұйымын көрсетеді. **Go to project** таңдаңыз.
3. Сол мәзірден **Tracing** таңдаңыз. **Traces** қойындысы соңғы күн трассаларын жаңасынан тізеді: кіріс, шығыс, кідіріс, токендері бар үш `course-assistant` трассасы.
4. Кірісі «Where does Litefuse keep its traces?» (Litefuse трассаларын қайда сақтайды?) деп сұрайтын трассаны таңдаңыз. Панель ағаш түрінде ашылады: `course-assistant`, астында `answer-question`, оның астында ұзақтығымен `search-docs` пен `llm-answer`. `llm-answer` таңдаңыз: панель промптты **System** және **User** хабарлары, жауапты **Assistant** хабары ретінде, `temperature: 0` мәнді `simulated-llm` моделін, `28 prompt → 17 completion (∑ 45)` токендерін және `$0.00079` құнын көрсетеді.
5. **Sessions** таңдаңыз: `session-1` — екі, `session-2` — бір трасса; құны мен токенін көру үшін кестені оңға жылжытыңыз. **Users** екі пайдаланушыны тізеді. Оқиға саны бақылау жазбаларын қамтиды, бөлек трасса саны емес; трасса санын **Sessions** не төмендегі SQL жаттығуымен салыстырыңыз. Кесте ұсақ құнды екі ондыққа дөңгелектейді, сондықтан `$0.00` тіркелген `$0.00153` сияқты құнды да білдіруі мүмкін. Толық дәлдікке сеанс егжей-тегжейін не SQL қолданыңыз. **Home** трассалар, модель/пайдаланушы бойынша құн және кідіріс процентильдерін сызады.

Жеңілдетілген қытай интерфейсін қалдырсаңыз, тиісті белгілер: **前往项目** (Go to project, жобаға өту), **追踪** (Tracing, трассалау), **会话** (Sessions, сеанстар), **用户** (Users, пайдаланушылар), **首页** (Home, басты бет).

Litefuse мұның бәрін 6-бөлімде сұраған кестелерден оқиды. Жария API-ы да солай: келесі ұяшық SDK-дағы сол API кілттерімен екінші трассаны оқиды.

In [ ]:
trace = lab.litefuse_trace(trace_ids[1])

<a id="unit-u.aefa3e4bb45e4c2893bdecfaeec87c93"></a>

**Күтілетін нәтиже**

`curl` командасы, кейін трассаның үш бақылау жазбасы: `SPAN` `answer-question`, `RETRIEVER` `search-docs`, моделі `simulated-llm`, пайдалануы `{'input': 28, 'output': 17, 'total': 45}`, құны `0.00079` болатын `GENERATION` `llm-answer`. Атауы трасса атын `course-assistant`, кідірісін шамамен 0.23 секунд және жалпы құнын `0.00079` USD көрсетеді: 6-бөлім Doris-тен тапқан сандар.

<a id="unit-u.a8c035932b25422387d1e3de359f83f4"></a>

<a id="8-how-litefuse-uses-doris"></a>

## 8. Litefuse Doris-ті қалай қолданады

Litefuse кестелері 4–6-модуль мүмкіндіктерін қолданады. Келесі ұяшық жоба кестелерінің кілт бағандарын тізеді.

In [ ]:
lab.sql("""
SELECT TABLE_NAME AS table_name, COLUMN_NAME AS key_column, COLUMN_KEY AS key_type
FROM information_schema.columns
WHERE TABLE_SCHEMA = 'litefuse'
  AND TABLE_NAME IN ('spans_llm_app', 'traces_scalar_llm_app')
  AND COLUMN_KEY <> ''
ORDER BY TABLE_NAME, ORDINAL_POSITION
""", title="Key columns of the project's tables");

<a id="unit-u.1a8e29fb080a42439423008aa0fae04c"></a>

**Күтілетін нәтиже**

Бес жол. `spans_llm_app` — `trace_id`, `start_time`, `span_id` кілтті Duplicate Key (`DUP`) кестесі: бақылау жазбалары тек қосылады, кілт трасса жолдарын уақыт ретімен бірге ұстайды. `traces_scalar_llm_app` — `id`, `start_time` кілтті Unique Key (`UNI`): Litefuse трасса жолын қайта жазса, 4-модульдегі `error_alerts` сияқты жаңа жол ескісін алмастырады.

Екеуі де күндік `AUTO PARTITION BY RANGE (date_trunc(start_time, 'day'))` арқылы бөлімделіп, трасса ID-імен `BUCKETS AUTO` ішіне таратылады (5-модуль); `SHOW CREATE TABLE litefuse.spans_llm_app` осы сөйлемдерді көрсетеді. Litefuse `spans_llm_app` үстінде әр трассаның токендері мен құнын күн бойынша қосатын `trace_metrics_agg_llm_app` синхронды көрінісін де құрады (6-модуль); `SHOW CREATE MATERIALIZED VIEW trace_metrics_agg_llm_app ON litefuse.spans_llm_app` соны көрсетеді.

<a id="unit-u.94d349ba0b8941778a08e8487cdf86dd"></a>

`spans_llm_app` ішінде 14 бағанға кері индекс бар (`SHOW INDEX FROM litefuse.spans_llm_app` тізеді). `input` пен `output` индекстері мәтінді сөздерге бөледі (`"parser" = "unicode"`, кіші әріппен), сондықтан әр жолды оқымай, сөзі бар промпт пен жауапты табуға болады. Келесі ұяшық кірістерден Litefuse сөзін іздейді.

In [ ]:
lab.sql("""
SELECT trace_id, name, type
FROM litefuse.spans_llm_app
WHERE input MATCH_ANY 'litefuse'
ORDER BY start_time, is_root DESC
""", title="Observations whose input contains the word litefuse");

<a id="unit-u.73e785710a4040b3bdd3e98d941a0965"></a>

**Күтілетін нәтиже**

5-бөлімді әр орындауға үш жол — Litefuse туралы трассаның бақылау жазбалары: сұрақ үшеуінің де кірісінде; `answer-question` пен `search-docs` оны аргумент ретінде, `llm-answer` промптында алған. `MATCH_ANY` толық сөздерге регистрсіз сәйкесіп, берілгендердің кез келгені болса жолды табады; `MATCH_ALL` бәрін талап етеді.

<a id="unit-u.d1e204ea45144facb29565df75c6ca7d"></a>

<a id="9-independent-exercise"></a>

## 9. Өзіндік жаттығу

Әр пайдаланушы үшін `spans_llm_app` ішінде трассаларды санап, модель шақыруларының токендері мен құнын қосыңыз. Шақыру токендері — `usage_details['total']` ішінде. Екі жол күтіңіз, алдымен `user-1`.

In [ ]:
# Your code here

<a id="unit-u.017829d2346f4bb4b19d6272fd3e48a3"></a>

Келесі ұяшықта үлгі шешім бар; ол жиналған. Салыстыру үшін ашыңыз немесе орындаңыз.

In [ ]:
lab.sql("""
SELECT user_id,
       COUNT(DISTINCT trace_id) AS traces,
       SUM(usage_details['total']) AS total_tokens,
       SUM(total_cost) AS total_cost_usd
FROM litefuse.spans_llm_app
WHERE type = 'GENERATION'
GROUP BY user_id
ORDER BY total_cost_usd DESC
""", title="Traces, tokens, and cost by user");

<a id="unit-u.f0ccd021dc7f40d88afe3cc20efcb28f"></a>

**Күтілетін нәтиже**

`user-1`: 2 трасса, 87 токен, 0.00153 USD; `user-2`: 1 трасса, 42 токен, 0.00074 USD. 5-бөлімнің әр орындалуы тағы сонша қосады. Litefuse **Sessions** беті сол жиынтықтарды көрсетеді. Тек generation жазбалары саналады: басқаларында токен не құн жоқ, әр трассада дәл бір generation бар; сондықтан олардың арасында трасса санау бәрін қамтиды.

<a id="unit-u.659316ab59b643bfa68199a57270f72a"></a>

<a id="completion-criteria"></a>

## Аяқтау шарттары

- LLM трассасы `otel_traces` span-ы тіркемейтін сұрау туралы нені тіркейтінін айта аласыз.
- `spans_llm_app` ішінде үш трассаңыздың бақылау жазбалары, әр модель шақыруының моделі, токені мен құны бар; сол трассаны Litefuse веб-интерфейсінен таптыңыз.
- Litefuse дерек жүктеуі үшін BE-ге неге жеке соңғы нүкте қажет болғанын түсіндіре аласыз.
- Сұрауыңыз токендер мен құнды пайдаланушы бойынша қосады.

<a id="clean-up"></a>

## Тазалау

Litefuse қосулы қалады, шамамен 1 GB жады қолданады. `lab.stop_litefuse()` контейнерлерін тоқтатып жояды; пайдаланушысы, кілттері, трассалары томдары мен Doris-те қалады, `lab.start_litefuse()` солармен қайта қосады. `lab.teardown()` Litefuse-ті қалған ортамен, томдарымен және `litefuse` дерекқорымен бірге жояды.

<a id="unit-u.98507a2ac88448d3a984e3c25123372c"></a>

<a id="next"></a>

## Әрі қарай

[7-тест: Трассаларды талдау және Litefuse](quiz07_analysis_and_litefuse.ipynb) файлын ашып, [8-модуль: Grafana көмегімен ескерту](../module08-alerting/course.md) бөліміне өтіңіз. [Қосымша 7A: Нақты модельді Litefuse арқылы трассалау](optional07_trace_a_real_model.ipynb) 5-бөлімді нақты модельмен қайталайды.